In [1]:
# ===================================================================
# SISTEMA DE ANÁLISIS AMBIENTAL PARA AULA 0.13
# ===================================================================
# AUTOR: Carmen Gómez Alarcón
# FECHA: 2026
# DESCRIPCIÓN:
#   - Análisis de deltas (cambios temporales) de CO2, Temperatura,
#     Humedad y Consumo vs Ocupación.
#   - Análisis de CO2 vs Ocupación en valores absolutos.
#   - Gráficos separados por meses (Febrero y Marzo) y por sensor.
# ===================================================================

import pandas as pd
import os
import glob
import matplotlib.pyplot as plt
import numpy as np

# Directorio de salida
OUTPUT_DIR = 'plots_scatters_deltas'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================================
# CARGAR TODOS LOS DATOS
# ==========================================
file_pattern = '013_data_*_week_*.xlsx'
all_files = sorted(glob.glob(file_pattern))

if not all_files:
    print(f"\nERROR: No files found matching '{file_pattern}'")
    exit()

print(f"\nFound {len(all_files)} weekly files")

metadata_cols = ['Room', 'Volume', 'Floor', 'Sensor_Location', 'Variable_Measure']

SENSORS   = ['Corridor', 'Window', 'Hanging', 'Lighting']
VARIABLES = ['Temperature', 'CO2', 'Humidity', 'Consumption']

sensor_colors = {
    'Corridor': '#2196F3',  # Azul
    'Window':   '#E91E63',  # Rosa
    'Hanging':  '#4CAF50',  # Verde
    'Lighting': '#FF9800',  # Naranja para consumo
}

month_order = ['February', 'March']

print("\nLOADING ALL DATA...")

all_records = []

for file_path in all_files:
    fname = os.path.basename(file_path)
    print(f"   {fname}...", end=" ")

    df = pd.read_excel(file_path)
    date_cols  = [c for c in df.columns if c not in metadata_cols]
    timestamps = pd.to_datetime(date_cols, errors='coerce')

    def get_series(variable, location):
        mask = (df['Variable_Measure'] == variable) & (df['Sensor_Location'] == location)
        row  = df[mask]
        if len(row) > 0:
            return row[date_cols].iloc[0].astype(float).values
        return np.full(len(date_cols), np.nan)

    occ_mask  = df['Variable_Measure'] == 'Occupancy'
    occ_row   = df[occ_mask]
    occupancy = occ_row[date_cols].iloc[0].astype(float).values if len(occ_row) > 0 else np.full(len(date_cols), np.nan)

    months = [t.month_name() if not pd.isnull(t) else None for t in timestamps]

    for sensor in SENSORS:
        for var in VARIABLES:
            values = get_series(var, sensor)
            for i, ts in enumerate(timestamps):
                if pd.isnull(ts):
                    continue
                all_records.append({
                    'timestamp': ts,
                    'month':     months[i],
                    'sensor':    sensor,
                    'variable':  var,
                    'value':     values[i],
                    'occupancy': occupancy[i],
                })

    print("OK")

df_all = pd.DataFrame(all_records)
df_all = df_all.sort_values('timestamp')

print(f"\n   Total records: {len(df_all)}")

# Verificar si hay datos de consumo
if 'Consumption' in df_all['variable'].unique():
    consumption_data = df_all[df_all['variable'] == 'Consumption']
    print(f"   Consumption data found: {len(consumption_data)} records")
    print(f"   Consumption range: {consumption_data['value'].min():.2f} - {consumption_data['value'].max():.2f} kWh")
else:
    print(f"   ⚠️ WARNING: No Consumption data found in files")

# ==========================================
# CALCULAR DELTAS (punto a punto, por sensor+variable)
# ==========================================
print("\nCalculating deltas...")

df_all['delta_value'] = df_all.groupby(['sensor', 'variable'])['value'].diff()

# Filtrar solo Feb y Mar
unique_months = [m for m in month_order if m in df_all['month'].unique()]
df_all = df_all[df_all['month'].isin(unique_months)]

print(f"   Records after filter (Feb+Mar): {len(df_all)}")

# ==========================================
# FUNCIÓN: ΔVariable vs Occupancy
# ==========================================
def scatter_delta(y_var, y_label, suptitle, filename, y_scale=None):
    month_pos = {m: i for i, m in enumerate(unique_months)}
    n_months  = len(unique_months)

    if n_months == 1:
        fig, axes = plt.subplots(1, 1, figsize=(7, 5))
        axes_flat = [axes]
    else:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        axes_flat = axes.flatten()

    fig.suptitle(suptitle, fontsize=14, fontweight='bold', y=1.02)

    any_data = False

    for month, pos in month_pos.items():
        ax = axes_flat[pos]
        ax.set_title(month, fontsize=12, fontweight='bold')
        ax.set_xlabel('Occupancy (persons)', fontsize=10)
        ax.set_ylabel(y_label, fontsize=10)
        ax.axhline(0, color='gray', linewidth=0.8, linestyle='-', alpha=0.5)
        ax.axvline(0, color='gray', linewidth=0.8, linestyle='-', alpha=0.5)
        ax.grid(True, alpha=0.25, linestyle='--')
        ax.tick_params(labelsize=9)
        
        if y_scale:
            if y_scale == 'log':
                ax.set_yscale('log')
            elif y_scale == 'symlog':
                ax.set_yscale('symlog')

        handles = []

        for sensor in SENSORS:
            sub = df_all[
                (df_all['month'] == month) &
                (df_all['sensor'] == sensor) &
                (df_all['variable'] == y_var)
            ].dropna(subset=['occupancy', 'delta_value'])

            if len(sub) == 0:
                continue

            any_data = True
            color = sensor_colors.get(sensor, '#999999')

            sc = ax.scatter(sub['occupancy'], sub['delta_value'],
                            c=color, alpha=0.45, s=10, edgecolors='none',
                            label=sensor)
            handles.append(sc)

            try:
                x_v = sub['occupancy'].values
                y_v = sub['delta_value'].values
                if len(x_v) > 1 and len(y_v) > 1:
                    z   = np.polyfit(x_v, y_v, 1)
                    p   = np.poly1d(z)
                    x_t = np.linspace(x_v.min(), x_v.max(), 100)
                    ax.plot(x_t, p(x_t), color=color, linewidth=1.6,
                            linestyle='--', alpha=0.9)
            except Exception:
                pass

        if handles:
            ax.legend(fontsize=9, markerscale=2, framealpha=0.7)

    if not any_data:
        print(f"   SKIPPED (no data): {filename}")
        plt.close()
        return

    plt.tight_layout()
    fp = os.path.join(OUTPUT_DIR, filename)
    plt.savefig(fp, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"   SAVED: {filename}")

# ==========================================
# GRÁFICO ESPECIAL PARA CONSUMO (DELTAS)
# ==========================================
def scatter_consumption_delta():
    month_pos = {m: i for i, m in enumerate(unique_months)}
    n_months  = len(unique_months)

    if n_months == 1:
        fig, axes = plt.subplots(1, 1, figsize=(7, 5))
        axes_flat = [axes]
    else:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        axes_flat = axes.flatten()

    fig.suptitle('ΔConsumption vs Occupancy — Classroom 0.13 (Lighting sensor)', 
                 fontsize=14, fontweight='bold', y=1.02)

    any_data = False

    for month, pos in month_pos.items():
        ax = axes_flat[pos]
        ax.set_title(month, fontsize=12, fontweight='bold')
        ax.set_xlabel('Occupancy (persons)', fontsize=10)
        ax.set_ylabel('ΔConsumption (kWh)', fontsize=10)
        ax.axhline(0, color='gray', linewidth=0.8, linestyle='-', alpha=0.5)
        ax.axvline(0, color='gray', linewidth=0.8, linestyle='-', alpha=0.5)
        ax.grid(True, alpha=0.25, linestyle='--')
        ax.tick_params(labelsize=9)

        sub = df_all[
            (df_all['month'] == month) &
            (df_all['sensor'] == 'Lighting') &
            (df_all['variable'] == 'Consumption')
        ].dropna(subset=['occupancy', 'delta_value'])

        if len(sub) > 0:
            any_data = True
            color = sensor_colors['Lighting']
            
            ax.scatter(sub['occupancy'], sub['delta_value'],
                      c=color, alpha=0.5, s=15, edgecolors='none',
                      label='Lighting (Consumption)')
            
            try:
                x_v = sub['occupancy'].values
                y_v = sub['delta_value'].values
                if len(x_v) > 1:
                    z = np.polyfit(x_v, y_v, 1)
                    p = np.poly1d(z)
                    x_t = np.linspace(x_v.min(), x_v.max(), 100)
                    ax.plot(x_t, p(x_t), color=color, linewidth=1.6,
                           linestyle='--', alpha=0.9)
                    
                    r2 = np.corrcoef(x_v, y_v)[0, 1] ** 2
                    ax.text(0.05, 0.95, f'R² = {r2:.3f}', 
                           transform=ax.transAxes, fontsize=9,
                           verticalalignment='top',
                           bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
            except Exception:
                pass
            
            ax.legend(fontsize=9, markerscale=2, framealpha=0.7)
            
            print(f"   {month}: Consumption Δ mean = {sub['delta_value'].mean():.3f} kWh")
            print(f"          Consumption Δ std  = {sub['delta_value'].std():.3f} kWh")

    if not any_data:
        print("   SKIPPED: No Consumption data available")
        plt.close()
        return

    plt.tight_layout()
    fp = os.path.join(OUTPUT_DIR, '20_dConsumption_vs_Occupancy.png')
    plt.savefig(fp, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"   SAVED: 20_dConsumption_vs_Occupancy.png")

# ==========================================
# GRÁFICO: CONSUMPTION vs OCCUPANCY (VALORES ABSOLUTOS)
# ==========================================
def plot_occupancy_vs_consumption_absolute():
    """Gráfico de dispersión: Ocupación vs Consumo Eléctrico (absoluto)"""
    
    print("\n   Loading absolute consumption data...")
    
    consumption_data = []
    
    for file_path in all_files:
        if 'February' in file_path:
            month = 'February'
        elif 'March' in file_path:
            month = 'March'
        else:
            continue
        
        df = pd.read_excel(file_path)
        
        occ_row = df[df['Variable_Measure'] == 'Occupancy']
        if occ_row.empty:
            continue
        
        cons_row = df[
            (df['Variable_Measure'] == 'Consumption') & 
            (df['Sensor_Location'] == 'Lighting')
        ]
        if cons_row.empty:
            continue
        
        date_cols = [c for c in df.columns if c not in metadata_cols]
        
        occ_values = occ_row.iloc[0][date_cols].values
        cons_values = cons_row.iloc[0][date_cols].values
        
        for occ, cons in zip(occ_values, cons_values):
            if pd.notna(occ) and pd.notna(cons):
                consumption_data.append({
                    'month': month,
                    'occupancy': int(occ),
                    'consumption': cons
                })
    
    if not consumption_data:
        print("   No valid Consumption-Occupancy pairs found")
        return
    
    df_plot = pd.DataFrame(consumption_data)
    
    months = sorted(df_plot['month'].unique())
    
    if len(months) == 1:
        fig, ax = plt.subplots(1, 1, figsize=(8, 6))
        axes = [ax]
    else:
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        axes = axes.flatten()
    
    fig.suptitle('Lighting Consumption vs Occupancy — Classroom 0.13\n(Absolute Values)', 
                 fontsize=14, fontweight='bold', y=1.02)
    
    for idx, month in enumerate(months):
        ax = axes[idx]
        month_data = df_plot[df_plot['month'] == month]
        
        ax.scatter(month_data['occupancy'], month_data['consumption'],
                  alpha=0.4, s=20, c='#FF9800', edgecolors='none',
                  label='Individual readings')
        
        if len(month_data) > 1:
            z = np.polyfit(month_data['occupancy'], month_data['consumption'], 1)
            p = np.poly1d(z)
            x_line = np.linspace(month_data['occupancy'].min(), month_data['occupancy'].max(), 100)
            ax.plot(x_line, p(x_line), 'r--', linewidth=1.5, alpha=0.8, label='Trend line')
            
            corr_matrix = np.corrcoef(month_data['occupancy'], month_data['consumption'])
            r2 = corr_matrix[0, 1] ** 2
            ax.text(0.05, 0.95, f'R² = {r2:.3f}', transform=ax.transAxes,
                   fontsize=10, verticalalignment='top',
                   bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        
        grouped = month_data.groupby('occupancy')['consumption'].agg(['mean', 'std', 'count'])
        if len(grouped) > 0:
            for occ in grouped.index:
                mean_val = grouped.loc[occ, 'mean']
                std_val = grouped.loc[occ, 'std']
                count = grouped.loc[occ, 'count']
                if count >= 2:
                    ax.errorbar(occ, mean_val, yerr=std_val, fmt='o', 
                               color='darkorange', markersize=8, capsize=5,
                               capthick=1.5, elinewidth=1.5, alpha=0.7,
                               label='Mean ± std' if occ == grouped.index[0] else '')
                else:
                    ax.plot(occ, mean_val, 'o', color='darkorange', markersize=8, alpha=0.7)
        
        ax.set_xlabel('Occupancy (persons)', fontsize=11)
        ax.set_ylabel('Lighting Consumption (kWh)', fontsize=11)
        ax.set_title(f'{month}', fontsize=12, fontweight='bold')
        ax.grid(True, alpha=0.25, linestyle='--')
        ax.legend(fontsize=9, loc='best')
        
        print(f"   {month}: n={len(month_data)} readings")
        print(f"          Mean consumption: {month_data['consumption'].mean():.3f} kWh")
        print(f"          Range: {month_data['consumption'].min():.3f} - {month_data['consumption'].max():.3f} kWh")
    
    plt.tight_layout()
    fp = os.path.join(OUTPUT_DIR, '21_Consumption_vs_Occupancy_absolute.png')
    plt.savefig(fp, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"   SAVED: 21_Consumption_vs_Occupancy_absolute.png")

# ==========================================
# GRÁFICO NUEVO: CO₂ vs OCCUPANCY (VALORES ABSOLUTOS)
# ==========================================
def plot_co2_vs_occupancy_absolute():
    """Gráfico de dispersión: CO₂ vs Ocupación (valores absolutos)"""
    
    print("\n   Loading absolute CO2 data...")
    
    # Recopilar datos de CO₂ absolutos
    co2_data = []
    
    for file_path in all_files:
        if 'February' in file_path:
            month = 'February'
        elif 'March' in file_path:
            month = 'March'
        else:
            continue
        
        df = pd.read_excel(file_path)
        date_cols = [c for c in df.columns if c not in metadata_cols]
        
        occ_row = df[df['Variable_Measure'] == 'Occupancy']
        if occ_row.empty:
            continue
        
        occ_values = occ_row.iloc[0][date_cols].values
        
        for sensor in ['Corridor', 'Window', 'Hanging']:
            co2_row = df[
                (df['Variable_Measure'] == 'CO2') & 
                (df['Sensor_Location'] == sensor)
            ]
            if co2_row.empty:
                continue
            
            co2_values = co2_row.iloc[0][date_cols].values
            
            for occ, co2 in zip(occ_values, co2_values):
                if pd.notna(occ) and pd.notna(co2):
                    co2_data.append({
                        'month': month,
                        'sensor': sensor,
                        'occupancy': occ,
                        'co2': co2
                    })
    
    if not co2_data:
        print("   No valid CO2-Occupancy pairs found")
        return
    
    df_plot = pd.DataFrame(co2_data)
    months = sorted(df_plot['month'].unique())
    
    if len(months) == 1:
        fig, axes = plt.subplots(1, 1, figsize=(8, 6))
        axes_flat = [axes]
    else:
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        axes_flat = axes.flatten()
    
    fig.suptitle('CO₂ vs Occupancy — Classroom 0.13\n(Absolute Values)', 
                 fontsize=14, fontweight='bold', y=1.02)
    
    for month, ax in zip(months, axes_flat):
        ax.set_title(month, fontsize=12, fontweight='bold')
        ax.set_xlabel('Occupancy (persons)', fontsize=11)
        ax.set_ylabel('CO₂ (ppm)', fontsize=11)
        ax.grid(True, alpha=0.25, linestyle='--')
        
        handles = []
        
        for sensor in ['Corridor', 'Window', 'Hanging']:
            sub = df_plot[
                (df_plot['month'] == month) &
                (df_plot['sensor'] == sensor)
            ].dropna(subset=['occupancy', 'co2'])
            
            if len(sub) == 0:
                continue
            
            color = sensor_colors.get(sensor, '#999999')
            
            sc = ax.scatter(sub['occupancy'], sub['co2'],
                            c=color, alpha=0.4, s=15, edgecolors='none',
                            label=sensor)
            handles.append(sc)
            
            try:
                x_v = sub['occupancy'].values
                y_v = sub['co2'].values
                if len(x_v) > 2:
                    z = np.polyfit(x_v, y_v, 1)
                    p = np.poly1d(z)
                    x_line = np.linspace(x_v.min(), x_v.max(), 100)
                    ax.plot(x_line, p(x_line), color=color, linewidth=1.6,
                            linestyle='--', alpha=0.8)
                    
                    r2 = np.corrcoef(x_v, y_v)[0, 1] ** 2
                    ax.text(0.05, 0.95 - 0.06 * ['Corridor','Window','Hanging'].index(sensor),
                            f'{sensor}: R² = {r2:.3f}',
                            transform=ax.transAxes, fontsize=8,
                            verticalalignment='top',
                            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
            except Exception:
                pass
        
        if handles:
            ax.legend(fontsize=9, markerscale=2, framealpha=0.7)
    
    plt.tight_layout()
    fp = os.path.join(OUTPUT_DIR, '22_CO2_vs_Occupancy_absolute.png')
    plt.savefig(fp, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"   SAVED: 22_CO2_vs_Occupancy_absolute.png")
    
    # Análisis de correlación
    print("\n   Correlation analysis (absolute CO2 vs Occupancy):")
    for sensor in ['Corridor', 'Window', 'Hanging']:
        sub = df_plot[df_plot['sensor'] == sensor].dropna(subset=['occupancy', 'co2'])
        if len(sub) > 5:
            corr = sub['occupancy'].corr(sub['co2'])
            print(f"      {sensor}: r = {corr:.3f} (n={len(sub)})")
        else:
            print(f"      {sensor}: Insufficient data (n={len(sub)})")

# ==========================================
# GENERAR TODOS LOS GRÁFICOS
# ==========================================
print("\n" + "="*60)
print("GENERATING DELTA SCATTER PLOTS")
print("="*60)

# Gráficos delta
scatter_delta('CO2',         'ΔCO₂ (ppm)',        'ΔCO₂ vs Occupancy — Classroom 0.13',         '17_dCO2_vs_Occupancy.png')
scatter_delta('Temperature', 'ΔTemperature (°C)', 'ΔTemperature vs Occupancy — Classroom 0.13', '18_dTemperature_vs_Occupancy.png')
scatter_delta('Humidity',    'ΔHumidity (%)',      'ΔHumidity vs Occupancy — Classroom 0.13',    '19_dHumidity_vs_Occupancy.png')

# Gráfico delta consumo
print("\n   Processing Consumption delta data...")
scatter_consumption_delta()

# Gráfico consumo absoluto
print("\n   Processing Consumption absolute data...")
plot_occupancy_vs_consumption_absolute()

# NUEVO: Gráfico CO₂ absoluto
print("\n   Processing CO2 absolute data...")
plot_co2_vs_occupancy_absolute()

# ==========================================
# ANÁLISIS DE CORRELACIONES (DELTAS)
# ==========================================
print("\n" + "="*60)
print("CORRELATION ANALYSIS (DELTAS)")
print("="*60)

def correlation_by_occupation(variable, min_occupancy=0):
    for sensor in SENSORS:
        sub = df_all[
            (df_all['sensor'] == sensor) &
            (df_all['variable'] == variable) &
            (df_all['occupancy'] >= min_occupancy)
        ].dropna(subset=['occupancy', 'delta_value'])
        
        if len(sub) > 5:
            corr = sub['occupancy'].corr(sub['delta_value'])
            print(f"   {sensor} - {variable}: r = {corr:.3f} (n={len(sub)})")

print("\nCorrelations (occupancy ≥ 0):")
for var in VARIABLES:
    correlation_by_occupation(var)

print("\nCorrelations (occupancy ≥ 3):")
for var in VARIABLES:
    correlation_by_occupation(var, min_occupancy=3)

print("\n" + "="*60)
print("ANALYSIS COMPLETE")
print("="*60)


Found 13 weekly files

LOADING ALL DATA...
   013_data_December_2025_week_4_days_22-31.xlsx... OK
   013_data_February_2026_week_1_days_1-7.xlsx... OK
   013_data_February_2026_week_2_days_8-14.xlsx... OK
   013_data_February_2026_week_3_days_15-21.xlsx... OK
   013_data_February_2026_week_4_days_22-28.xlsx... OK
   013_data_January_2026_week_1_days_1-7.xlsx... OK
   013_data_January_2026_week_2_days_8-14.xlsx... OK
   013_data_January_2026_week_3_days_15-21.xlsx... OK
   013_data_January_2026_week_4_days_22-31.xlsx... OK
   013_data_March_2026_week_1_days_1-7.xlsx... OK
   013_data_March_2026_week_2_days_8-14.xlsx... OK
   013_data_March_2026_week_3_days_15-21.xlsx... OK
   013_data_March_2026_week_4_days_22-31.xlsx... OK

   Total records: 755056
   Consumption data found: 188764 records
   Consumption range: 0.00 - 2.12 kWh

Calculating deltas...
   Records after filter (Feb+Mar): 601328

GENERATING DELTA SCATTER PLOTS
   SAVED: 17_dCO2_vs_Occupancy.png
   SAVED: 18_dTemperature_vs